# 04. Тестирование RuBERT NER baseline

Ноутбук загружает лучший validation-checkpoint, выполняет document-level BIO-декодирование на test и сохраняет `test_metrics.json` и `predictions.jsonl`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import runpy

PROJECT_DIR = Path("/content/drive/MyDrive/NER_RuREBus_project")
EXPERIMENT_CONFIG = PROJECT_DIR / "configs" / "experiments" / "ner_baseline_v1.yaml"
BEST_CHECKPOINT = PROJECT_DIR / "results" / "ner_baseline" / "ner_baseline_v1" / "checkpoints" / "best"
BOOTSTRAP = PROJECT_DIR / "colab_bootstrap.py"

for required_path in (EXPERIMENT_CONFIG, BEST_CHECKPOINT, BOOTSTRAP):
    if not required_path.exists():
        raise FileNotFoundError(f"Не найден {required_path}. Проверьте PROJECT_DIR и наличие checkpoint.")

bootstrap_project = runpy.run_path(str(BOOTSTRAP))["bootstrap_project"]
bootstrap_project(PROJECT_DIR)

In [ ]:
from rurebus_ie.training import test_ner_experiment

result = test_ner_experiment(EXPERIMENT_CONFIG, project_root=PROJECT_DIR)
print(f"Test loss: {result.loss:.4f}")
print(f"Strict precision: {result.metrics.precision:.4f}")
print(f"Strict recall: {result.metrics.recall:.4f}")
print(f"Strict micro-F1: {result.metrics.micro_f1:.4f}")
print(f"Strict macro-F1: {result.metrics.macro_f1:.4f}")

In [ ]:
import pandas as pd

per_class = pd.DataFrame(result.metrics.per_class).T
display(per_class.sort_values("f1", ascending=False))

sample_document_id = next(iter(result.predictions))
print("Пример документа:", sample_document_id)
display(pd.DataFrame([entity.__dict__ for entity in result.predictions[sample_document_id]]).head(30))